In [0]:
%sql
USE CATALOG dbacademy;
USE SCHEMA default;

## Generate Data

In [0]:
from pyspark.sql.functions import *

df = (
  spark
    .range(0, 60, 1, 1)
    .select(
      'id',
      (col('id') % 1000).alias('device_id'),
      (rand() * 100).alias('temperature_F')
    )
)

df.display()

In [0]:
df.write.saveAsTable('device_data')

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

@udf("double")
def F_to_Celsius(f):
    # Let's pretend some fancy math takes one second per row
    time.sleep(1)
    return (f -32) * (5/9)

celcius_df = (
    spark.table('device_data')
        .withColumn("celcius", F_to_Celsius(col('temperature_F')))
)

celcius_df.write.mode('overwrite').saveAsTable('celcius')

## Parallelize

In [0]:

# Repartition across the number of cores in your cluster
num_cores = 2 # 8 GB 2 cores current cluster

@udf("double")
def F_to_Celsius(f):
    # Let's pretend some fancy math takes one second per row
    time.sleep(1)
    return (f -32) * (5/9)

celcius_df = (
    spark.table('device_data')
        .repartition(num_cores)
        .withColumn("celcius", F_to_Celsius(col('temperature_F')))
)

celcius_df.write.mode('overwrite').saveAsTable('celcius')

In [0]:
celcius_df.explain()

## SQL UDFS

In [0]:
%sql
DROP FUNCTION IF EXISTS farh_to_cels;

CREATE FUNCTION farh_to_cels(farh DOUBLE)
  RETURNS DOUBLE
  RETURN ((farh - 32) * 5/9);

CREATE OR REPLACE TABLE celsius_sql AS
  SELECT farh_to_cels(temperature_F) as Farh_tocels_convert FROM device_data;

In [0]:
%sql
SELECT * FROM celsius_sql;

In [0]:
%sql
explain SELECT farh_to_cels(temperature_F) as Farh_to_cels_convert FROM device_data;